In [1]:
import numpy as np
import pandas as pd
from numpy import ndarray

In [2]:
data = pd.read_csv('housing.csv')

In [3]:
data.shape

(20640, 10)

In [4]:
data.head(5)

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [5]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           20640 non-null  float64
 1   latitude            20640 non-null  float64
 2   housing_median_age  20640 non-null  float64
 3   total_rooms         20640 non-null  float64
 4   total_bedrooms      20433 non-null  float64
 5   population          20640 non-null  float64
 6   households          20640 non-null  float64
 7   median_income       20640 non-null  float64
 8   median_house_value  20640 non-null  float64
 9   ocean_proximity     20640 non-null  str    
dtypes: float64(9), str(1)
memory usage: 1.6 MB


In [6]:
data.isnull().sum()

longitude               0
latitude                0
housing_median_age      0
total_rooms             0
total_bedrooms        207
population              0
households              0
median_income           0
median_house_value      0
ocean_proximity         0
dtype: int64

In [7]:
#drop nan
data = data.dropna()
data.isnull().sum()

longitude             0
latitude              0
housing_median_age    0
total_rooms           0
total_bedrooms        0
population            0
households            0
median_income         0
median_house_value    0
ocean_proximity       0
dtype: int64

In [8]:
# convert str to number (onehot) -> df_encoded

In [9]:
data["ocean_proximity"].unique()

<StringArray>
['NEAR BAY', '<1H OCEAN', 'INLAND', 'NEAR OCEAN', 'ISLAND']
Length: 5, dtype: str

In [10]:
df_encoded = pd.get_dummies(data, columns=["ocean_proximity"], dtype= int)

In [11]:
df_encoded.head(5)

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity_<1H OCEAN,ocean_proximity_INLAND,ocean_proximity_ISLAND,ocean_proximity_NEAR BAY,ocean_proximity_NEAR OCEAN
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,0,0,0,1,0
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,0,0,0,1,0
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,0,0,0,1,0
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,0,0,0,1,0
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,0,0,0,1,0


In [12]:
df_encoded.isnull().sum()

longitude                     0
latitude                      0
housing_median_age            0
total_rooms                   0
total_bedrooms                0
population                    0
households                    0
median_income                 0
median_house_value            0
ocean_proximity_<1H OCEAN     0
ocean_proximity_INLAND        0
ocean_proximity_ISLAND        0
ocean_proximity_NEAR BAY      0
ocean_proximity_NEAR OCEAN    0
dtype: int64

In [13]:
df_encoded.shape

(20433, 14)

In [14]:
#split train and test

In [47]:
x = df_encoded.drop(columns=["median_house_value"]).values
y = df_encoded[["median_house_value"]].values

indices = np.arange(x.shape[0])
np.random.shuffle(indices)

split_inx = int(0.8*indices.shape[0])

train , test = indices[:split_inx], indices[split_inx:]
x_train , x_test = x[train], x[test]
y_train , y_test = y[train], y[test]

In [48]:
print(f"X_train shape: {x_train.shape}")
print(f"Y_train shape: {y_train.shape}")

X_train shape: (16346, 13)
Y_train shape: (16346, 1)


In [49]:
print(f"X_train shape: {x_test.shape}")
print(f"Y_train shape: {y_test.shape}")

X_train shape: (4087, 13)
Y_train shape: (4087, 1)


# start deeplearning with numpy


In [18]:
rng = np.random.default_rng(42)

## activation ReLU

In [19]:
def relu(relu_input)-> np.ndarray:
    return np.maximum(0, relu_input)

## He initialaze

In [20]:
def he_initialize(num_inputs,num_out)-> np.ndarray:
    std = np.sqrt(2 / num_inputs)
    return rng.normal(0,std,size=(num_inputs,num_out))

## MSE LOSS

In [21]:
def mse_loss(mse_loss_x ,mse_loss_y) ->np.ndarray:
    return np.sum(np.square(mse_loss_x - mse_loss_y))

## Batch Normalization

In [22]:
#تفاوت اصلیِ Batch Normalization در دیپ‌لرنینگ با نرمال‌سازی ساده، این است که ما اجازه می‌دهیم شبکه خودش تصمیم بگیرد آیا واقعاً می‌خواهد خروجی‌اش نرمال باشد یا نه! برای این کار دو پارامتر قابل یادگیری به اسم Gamma ($\gamma$) و Beta ($\beta$) اضافه می‌کنیم.

In [23]:
#استفاده از var به جای std: در فرمول اصلی Batch Norm که در مقالات علمی (مثل مقاله اورجینال Sergey Ioffe) آمده، معمولاً از واریانس (var) استفاده می‌شود. اگر از std استفاده کنی، در مرحله Backward (که می‌خواهی مشتق بگیری)، فرمول ریاضی کمی پیچیده‌تر می‌شود. اگر از var استفاده کنی، مشتق‌گیری تمیزتر است.

In [24]:
def batch_norm(batch_norm_input,gamma,beta,eps=1e-5)-> np.ndarray:
    mean = np.mean(batch_norm_input,axis=0)
    var = np.var(batch_norm_input,axis=0)
    x_hat = (batch_norm_input - mean) / np.sqrt(var + eps)
    return gamma * x_hat + beta

## dropout

In [ ]:
# تو خود تابع dropout  اومد مشکل  تغییر میانگین را حل کرد !!

In [38]:
def dropout(arr,p=0.5,training=True)-> np.ndarray:
    if not training or p == 0:
        return arr

    mask = (rng.random(arr.shape) > p).astype(int)
    return (arr * mask) / (1-p)

In [ ]:
# بایاس مستقیماً با خودِ «ماتریس وزن» جمع نمی‌شود. ابتدا ضرب ماتریسی داده‌ها در وزن انجام می‌شود، سپس بایاس به حاصلِ آن ضرب اضافه می‌شود و در نهایت کل این مجموعه وارد تابع اکتیویشن (ReLU) می‌شود.
#ترتیب میشه ضرب وزن در ورودی لایه + بایاس میره تو اکتیویشن

In [46]:
class NeuralNetwork:
    def __init__(self,input_dim,output_dim,hidden_layers,training_data,testing_data):
        self.wight_list = []
        self.bias_list = []
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.hidden_layers = hidden_layers
        self.training_data = training_data
        self.testing_data = testing_data
        self.initialize_weights(he_initializer = True)

    def initialize_weights(self,he_initializer = True):
        if he_initializer:
            self.wight_list.append(he_initialize(self.input_dim,self.hidden_layers[0]))

            for index in range(len(self.hidden_layers)-1):
                self.wight_list.append(he_initialize(self.hidden_layers[index],self.hidden_layers[index+1]))
                self.bias_list.append(np.zeros((1,self.hidden_layers[index])))

            self.bias_list.append(np.zeros((1,self.hidden_layers[-1])))
            self.wight_list.append(he_initialize(self.hidden_layers[-1],self.output_dim))
            self.bias_list.append(np.zeros((1,self.output_dim)))

        else:
            self.wight_list.append(rng.normal(0,1,size=(self.input_dim,self.hidden_layers[0])))

            for index in range(len(self.hidden_layers)-1):
                self.wight_list.append(rng.normal(0,1,size=(self.hidden_layers[index],self.hidden_layers[index+1])))
                self.bias_list.append(np.zeros((1,self.hidden_layers[index])))

            self.bias_list.append(np.zeros((1,self.hidden_layers[-1])))
            self.wight_list.append(rng.normal(0,1,size=(self.hidden_layers[-1],self.output_dim)))
            self.bias_list.append(np.zeros((1,self.output_dim)))


    def forward(self,x):
        self.z_list = []
        self.a_list = []

        current_input = x

        for index in range(len(self.wight_list)-1):
            z = current_input @ self.wight_list[index] + self.bias_list[index]
            self.z_list.append(z)

            a = relu(self.z_list[index])
            self.a_list.append(a)

            current_input = self.a_list[index]

        last_z = current_input @ self.wight_list[-1] + self.bias_list[-1]
        self.z_list.append(last_z)
        self.a_list.append(last_z)
        return self.a_list[-1]



    def back_propagation(self,y_target):
        ...